In [4]:
import cdx_toolkit
import trafilatura
from datetime import datetime

# Initialize the Common Crawl CDX client
cdx = cdx_toolkit.CDXFetcher(source='cc')

print("Setup complete.")
print(f"cdx_toolkit version: {cdx_toolkit.__version__}")
print(f"trafilatura version: {trafilatura.__version__}")

Setup complete.
cdx_toolkit version: 0.9.38
trafilatura version: 2.0.0


In [5]:
# Target newspaper domains, mixed by size and state
# Format: domain -> (state, size_category)

TARGET_PAPERS = {
    # Large national/major metro
    'nytimes.com':           ('NY', 'large'),
    'washingtonpost.com':    ('DC', 'large'),
    'latimes.com':           ('CA', 'large'),
    'chicagotribune.com':    ('IL', 'large'),
    
    # Medium regional
    'desmoinesregister.com': ('IA', 'medium'),
    'azcentral.com':         ('AZ', 'medium'),
    'tennessean.com':        ('TN', 'medium'),
    'startribune.com':       ('MN', 'medium'),
    
    # Smaller local
    'argusleader.com':       ('SD', 'small'),
    'jacksonsun.com':        ('TN', 'small'),
    'rgj.com':               ('NV', 'small'),
}

print(f"Targeting {len(TARGET_PAPERS)} newspaper domains")
print(f"States covered: {sorted(set(s for s, _ in TARGET_PAPERS.values()))}")
print(f"Size breakdown:")
for size in ['large', 'medium', 'small']:
    papers = [d for d, (_, sz) in TARGET_PAPERS.items() if sz == size]
    print(f"  {size}: {len(papers)} ({', '.join(papers)})")

Targeting 11 newspaper domains
States covered: ['AZ', 'CA', 'DC', 'IA', 'IL', 'MN', 'NV', 'NY', 'SD', 'TN']
Size breakdown:
  large: 4 (nytimes.com, washingtonpost.com, latimes.com, chicagotribune.com)
  medium: 4 (desmoinesregister.com, azcentral.com, tennessean.com, startribune.com)
  small: 3 (argusleader.com, jacksonsun.com, rgj.com)


In [ ]:
# For each target paper, query Common Crawl to see how many pages it has indexed
# We'll limit to a small sample year (2023) to keep it fast for the demo

coverage_results = {}

for domain, (state, size) in TARGET_PAPERS.items():
    print(f"\nQuerying: {domain} ({state}, {size})...", end=" ", flush=True)
    
    try:
        # Query CDX for this domain in 2023
        # The url pattern uses /* to match all pages on the domain
        url_pattern = f"{domain}/*"
        
        # Count results without fetching content
        count = 0
        for obj in cdx.iter(
            url_pattern,
            from_ts='20230101',
            to_ts='20231231',
            limit=100  # cap at 100 for the demo
        ):
            count += 1
        
        coverage_results[domain] = {
            'state': state,
            'size': size,
            'pages_2023': count
        }
        print(f"{count} pages found")
        
    except Exception as e:
        print(f"ERROR: {e}")
        coverage_results[domain] = {
            'state': state,
            'size': size,
            'pages_2023': None,
            'error': str(e)
        }

print("\n--- Coverage Summary ---")
for domain, info in coverage_results.items():
    pages = info['pages_2023']
    status = f"{pages} pages" if pages is not None else "FAILED"
    print(f"  {domain:30s} {info['state']:3s} {info['size']:7s} {status}")


Querying: nytimes.com (NY, large)... 

11 failures for url https://index.commoncrawl.org/CC-MAIN-2019-43-index {'to_ts': '20231231', 'limit': 65, 'url': 'nytimes.com/*', 'output': 'json', 'page': 15, 'from': '20230101'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
12 failures for url https://index.commoncrawl.org/CC-MAIN-2019-43-index {'to_ts': '20231231', 'limit': 65, 'url': 'nytimes.com/*', 'output': 'json', 'page': 15, 'from': '20230101'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
13 failures for url https://index.commoncrawl.org/CC-MAIN-2019-43-index {'to_ts': '20231231', 'limit': 65, 'url': 'nytimes.com/*', 'output': 'json', 'page': 15, 'from': '20230101'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
14 failures for url https://index.commoncrawl.org/CC-MAIN-2019-43-index {'to_ts': '20231231', 'limit': 65, 'url': 'nytimes.com/*', 'output': 'json', 'page': 15, 'from': '20230

In [ ]:
# Force cdx_toolkit to query only one specific recent crawl
# This avoids cycling through old, unreliable indexes
# CC-MAIN-2024-46 is a November 2024 crawl - recent and should be stable

coverage_results = {}
SPECIFIC_CRAWL = 'CC-MAIN-2024-46'

print(f"Querying crawl: {SPECIFIC_CRAWL}")
print(f"This may take 2-5 minutes...\n")

for domain, (state, size) in TARGET_PAPERS.items():
    print(f"Querying: {domain:30s} ({state}, {size:7s})...", end=" ", flush=True)
    
    try:
        url_pattern = f"{domain}/*"
        count = 0
        
        # Restrict to one specific crawl using cc_indexes
        for obj in cdx.iter(
            url_pattern,
            limit=50,
            cc_indexes=[SPECIFIC_CRAWL]
        ):
            count += 1
        
        coverage_results[domain] = {
            'state': state,
            'size': size,
            'pages_found': count
        }
        print(f"{count} pages")
        
    except Exception as e:
        print(f"ERROR: {type(e).__name__}")
        coverage_results[domain] = {
            'state': state,
            'size': size,
            'pages_found': None,
            'error': str(e)[:100]
        }

print("\n--- Coverage Summary ---")
print(f"{'Domain':<30s} {'State':5s} {'Size':8s} {'Result':<15s}")
print("-" * 60)
for domain, info in coverage_results.items():
    pages = info['pages_found']
    status = f"{pages} pages" if pages is not None else "FAILED"
    print(f"{domain:<30s} {info['state']:<5s} {info['size']:<8s} {status:<15s}")

Querying crawl: CC-MAIN-2024-46
This may take 2-5 minutes...

Querying: nytimes.com                    (NY, large  )... 

11 failures for url https://index.commoncrawl.org/CC-MAIN-2020-29-index {'limit': 20, 'cc_indexes': ['CC-MAIN-2024-46'], 'url': 'nytimes.com/*', 'output': 'json', 'page': 2, 'from': '20250529122540'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
12 failures for url https://index.commoncrawl.org/CC-MAIN-2020-29-index {'limit': 20, 'cc_indexes': ['CC-MAIN-2024-46'], 'url': 'nytimes.com/*', 'output': 'json', 'page': 2, 'from': '20250529122540'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
13 failures for url https://index.commoncrawl.org/CC-MAIN-2020-29-index {'limit': 20, 'cc_indexes': ['CC-MAIN-2024-46'], 'url': 'nytimes.com/*', 'output': 'json', 'page': 2, 'from': '20250529122540'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
14 failures for url https://index.commoncrawl.org/CC-MAIN-2020-29-index {'limit': 20, 'cc_indexes': ['CC-MAIN-2024-46'

ERROR: ValueError
Querying: washingtonpost.com             (DC, large  )... 

11 failures for url https://index.commoncrawl.org/CC-MAIN-2008-2009-index {'limit': 50, 'cc_indexes': ['CC-MAIN-2024-46'], 'url': 'washingtonpost.com/*', 'output': 'json', 'page': 0, 'from': '20250529141907'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
12 failures for url https://index.commoncrawl.org/CC-MAIN-2008-2009-index {'limit': 50, 'cc_indexes': ['CC-MAIN-2024-46'], 'url': 'washingtonpost.com/*', 'output': 'json', 'page': 0, 'from': '20250529141907'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
13 failures for url https://index.commoncrawl.org/CC-MAIN-2008-2009-index {'limit': 50, 'cc_indexes': ['CC-MAIN-2024-46'], 'url': 'washingtonpost.com/*', 'output': 'json', 'page': 0, 'from': '20250529141907'}: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
14 failures for url https://index.commoncrawl.org/CC-MAIN-2008-2009-index {'limit': 50, 'cc

In [3]:
import requests

# Direct query to a single specific Common Crawl index
# Bypassing cdx_toolkit entirely

INDEX = 'CC-MAIN-2024-46'
TEST_DOMAIN = 'desmoinesregister.com'

url = f"https://index.commoncrawl.org/{INDEX}-index"
params = {
    'url': f'{TEST_DOMAIN}/*',
    'output': 'json',
    'limit': 10
}

print(f"Querying: {url}")
print(f"Params: {params}\n")

try:
    response = requests.get(url, params=params, timeout=30)
    print(f"Status: {response.status_code}")
    print(f"Response size: {len(response.text)} chars")
    print(f"\nFirst 500 chars of response:")
    print(response.text[:500])
except requests.exceptions.RequestException as e:
    print(f"Request failed: {type(e).__name__}: {e}")

Querying: https://index.commoncrawl.org/CC-MAIN-2024-46-index
Params: {'url': 'desmoinesregister.com/*', 'output': 'json', 'limit': 10}

Request failed: ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
